<a href="https://colab.research.google.com/github/Medhini-dotcom/Well-Logging-Explorer/blob/main/Well_Logging_Explorer_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 🎓 Student's Guide: Interactive Petrophysics Laboratory

#### 1. What is this App?
From our perspective as engineering students, this dashboard is a sandbox to understand how raw subsurface logging curves translate into commercial reservoir potential. Instead of manually crunching equations cell-by-cell or looking at static charts in a textbook, this lab simulates real-time downhole measurements ($0$ to $3000\text{ m}$) with an integrated Archie Solver.

#### 2. Key Physics & Calculations We Are Observing:
* **Shale Volume ($V_{sh}$)**: Computed using the Gamma Ray track. High API values correspond to high shale content ($V_{sh} \rightarrow 1$), meaning clay minerals are clogging our reservoir. We can change the **Shale Cutoff** slider to see what counts as reservoir rock versus non-reservoir rock.
* **Effective Porosity ($\Phi_e$)**: Calculated using $\Phi_e = \Phi_{total} \times (1 - V_{sh})$. It represents the interconnected pore space available to store fluids. Notice how $\Phi_e$ drops to $0\%$ when drilling through pure shales!
* **Water Saturation ($S_w$)**: Solved via the **Archie Equation** in real-time:
  $$S_w = \sqrt{\frac{R_w}{\Phi_e^m \cdot R_t}}$$
  Adjusting the Water Resistivity ($R_w$) and Cementation Exponent ($m$) modifies this calculation. When $S_w$ is low (typically $< 35\%$) and the rock is clean, we are in a **Hydrocarbon Pay Zone** (the green band on Track 3).

#### 3. Navigating the Crossplot Space:
* **Red Cluster (Shales)**: Located in the high shale region where the raw porosity is high but the effective porosity and resistivity are extremely low.
* **Blue Cluster (Wet Reservoir)**: Clean sandstones/limestones with high porosity but saturated with salty formation water ($S_w \approx 100\%$), resulting in low resistivity.
* **Green Cluster (Pay Candidate)**: The sweet spot! These points have high effective porosity and high resistivity, showing where trapped hydrocarbons are blocking the electrical current.

In [ ]:
from IPython.display import HTML, display

display(HTML(r"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>IITB Advanced Formation Evaluator</title>
    <style>
        body { font-family: 'Segoe UI', system-ui, sans-serif; background: #030712; margin: 0; padding: 15px; color: #f3f4f6; }
        .dashboard { max-width: 1100px; margin: auto; background: #111827; padding: 20px; border-radius: 16px; border: 1px solid #1f2937; box-shadow: 0 10px 40px rgba(0,0,0,0.6); }
        .header-bar { display: flex; justify-content: space-between; align-items: center; border-bottom: 2px solid #1f2937; padding-bottom: 12px; margin-bottom: 15px; }
        .grid-layout { display: grid; grid-template-columns: 1.3fr 1fr; gap: 20px; }
        .panel { background: #1f2937; border-radius: 12px; padding: 15px; border: 1px solid #374151; }
        canvas { background: #070a13; border-radius: 8px; width: 100%; height: auto; }
        .controls-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 15px; margin-top: 15px; }
        .slider-group { background: #111827; padding: 10px 14px; border-radius: 8px; border: 1px solid #374151; }
        .slider-group label { display: flex; justify-content: space-between; font-size: 12px; color: #9ca3af; margin-bottom: 4px; }
        input[type=range] { accent-color: #06b6d4; width: 100%; height: 5px; cursor: pointer; }
        .badge { background: #0e7490; color: #e0f2fe; padding: 4px 10px; border-radius: 30px; font-size: 12px; font-weight: 600; text-transform: uppercase; }
        .metrics-bar { display: grid; grid-template-columns: repeat(4, 1fr); gap: 10px; margin-top: 15px; }
        .metric-card { background: #111827; padding: 12px; border-radius: 8px; border: 1px solid #374151; text-align: left; }
        .metric-card span { display: block; color: #9ca3af; font-size: 10px; text-transform: uppercase; letter-spacing: 0.05em; }
        .metric-card b { font-size: 18px; color: #06b6d4; }
    </style>
</head>
<body>

<div class="dashboard">
    <div class="header-bar">
        <div style="text-align: left;">
            <h2 style="margin: 0; color: #06b6d4; font-size: 20px;">IITB Petrophysics Lab: Multi-Track Evaluator & Crossplot</h2>
            <p style="color: #9ca3af; font-size: 12px; margin: 3px 0 0 0;">Dynamic Reservoir Characterization, Archie Solvers & Lithology Cluster Space</p>
        </div>
        <div>
            <span class="badge" id="zoneClassification">Evaluating Reservoir</span>
        </div>
    </div>

    <div class="grid-layout">
        <!-- Left Panel: Interactive Well Log Tracks -->
        <div class="panel">
            <h3 style="margin: 0 0 10px 0; font-size: 14px; color: #38bdf8; text-transform: uppercase;">Logs and Volumetrics Track</h3>
            <canvas id="logCanvas" width="550" height="380"></canvas>
        </div>

        <!-- Right Panel: Crossplot Space & Archie Inputs -->
        <div class="panel" style="display: flex; flex-direction: column; justify-content: space-between;">
            <div>
                <h3 style="margin: 0 0 10px 0; font-size: 14px; color: #38bdf8; text-transform: uppercase;">Neutron-Density Crossplot Space</h3>
                <canvas id="crossplotCanvas" width="380" height="220"></canvas>
            </div>

            <div class="controls-grid">
                <div class="slider-group">
                    <label><span>Probe Depth</span><b id="depthVal">1200 m</b></label>
                    <input type="range" id="depthRange" min="0" max="3000" step="20" value="1200">
                </div>
                <div class="slider-group">
                    <label><span>Water Res ($R_w$)</span><b id="rwVal">0.05 Ωm</b></label>
                    <input type="range" id="rwRange" min="0.01" max="0.5" step="0.01" value="0.05">
                </div>
                <div class="slider-group">
                    <label><span>Archie exponent (m)</span><b id="mVal">2.0</b></label>
                    <input type="range" id="mRange" min="1.5" max="2.5" step="0.1" value="2.0">
                </div>
                <div class="slider-group">
                    <label><span>Shale Cutoff ($V_{sh}$)</span><b id="vshCutoffVal">35%</b></label>
                    <input type="range" id="vshCutoffRange" min="10" max="70" step="5" value="35">
                </div>
            </div>
        </div>
    </div>

    <div class="metrics-bar">
        <div class="metric-card"><span>Selected Formation</span><b id="lithVal">Sandstone</b></div>
        <div class="metric-card"><span>Shale Volume ($V_{sh}$)</span><b id="vshVal">15.2%</b></div>
        <div class="metric-card"><span>Effective Porosity ($\Phi_e$)</span><b id="porVal">18.4%</b></div>
        <div class="metric-card"><span>Water Saturation ($S_w$)</span><b id="swVal">30.1%</b></div>
    </div>
</div>

<script>
    const logCanvas = document.getElementById('logCanvas');
    const logCtx = logCanvas.getContext('2d');

    const crossplotCanvas = document.getElementById('crossplotCanvas');
    const crossplotCtx = crossplotCanvas.getContext('2d');

    // Simulation parameters
    let currentDepth = 1200;
    let rw = 0.05;
    let archieM = 2.0;
    let vshCutoff = 0.35;

    // Synthetic continuous formations database
    function getSubsurfaceParameters(z) {
        // Continuous physical waveforms matching standard sedimentary sequences
        let baseGr = 50 + Math.sin(z * 0.008) * 25 + Math.cos(z * 0.002) * 35;
        let baseRes = 1.2 + Math.pow(Math.abs(Math.sin(z * 0.004) * 6.5), 1.5) + (z > 1000 && z < 1600 ? 55 * Math.sin((z - 1000) * Math.PI / 600) : 0);
        let basePhi = 0.32 - (baseRes > 5 ? 0.08 : 0.0) + Math.cos(z * 0.005) * 0.04;

        // Boundary adjustments
        let lith = 'Shale';
        if (z >= 800 && z < 1800) {
            lith = 'Sandstone';
            baseGr -= 35;
        } else if (z >= 1800) {
            lith = 'Limestone';
            baseGr -= 45;
            baseRes += 10;
            basePhi -= 0.1;
        }

        let gr = Math.max(15, Math.min(150, baseGr));
        let res = Math.max(0.1, Math.min(120, baseRes));
        let phi = Math.max(0.01, Math.min(0.42, basePhi));

        // Petrophysics equations
        let grMin = 20, grMax = 145;
        let vsh = Math.max(0, Math.min(1, (gr - grMin) / (grMax - grMin)));
        let phie = Math.max(0, phi * (1 - vsh));

        // Archie Solver
        let sw = 1.0;
        if (phie > 0.02) {
            sw = Math.min(1.0, Math.sqrt((rw / Math.pow(phie, archieM)) / res));
        }

        return { z, lith, gr, res, phi, vsh, phie, sw };
    }

    function drawLogs() {
        logCtx.clearRect(0, 0, logCanvas.width, logCanvas.height);
        let maxD = 3000;
        let w = logCanvas.width, h = logCanvas.height;
        let topPadding = 30, botPadding = 20;
        let plotHeight = h - topPadding - botPadding;

        // Log tracks layouts
        logCtx.strokeStyle = '#374151';
        logCtx.lineWidth = 1;
        logCtx.strokeRect(40, topPadding, 130, plotHeight);  // Track 1: GR
        logCtx.strokeRect(185, topPadding, 130, plotHeight); // Track 2: RT
        logCtx.strokeRect(330, topPadding, 180, plotHeight); // Track 3: Volumetric Integration

        // Titles
        logCtx.fillStyle = '#9ca3af';
        logCtx.font = 'bold 9px monospace';
        logCtx.fillText('GR (API)', 80, 18);
        logCtx.fillText('RESISTIVITY', 220, 18);
        logCtx.fillText('VOLUMETRIC MODEL', 375, 18);

        // Plot Curves
        for (let y = 0; y < plotHeight; y += 2) {
            let z = (y / plotHeight) * maxD;
            let p = getSubsurfaceParameters(z);

            // Track 1: GR curve
            let grX = 40 + (p.gr / 150) * 130;
            logCtx.fillStyle = '#eab308';
            logCtx.fillRect(grX, topPadding + y, 2, 2);

            // Track 2: Resistivity (Logarithmic approximation display)
            let rtX = 185 + (Math.log10(p.res + 0.9) / 2.1) * 130;
            logCtx.fillStyle = '#06b6d4';
            logCtx.fillRect(rtX, topPadding + y, 2, 2);

            // Track 3: Volumetric shading representing Shale vs Sand vs Fluid
            let shaleW = p.vsh * 180;
            let porW = p.phie * 180;
            let matrixW = 180 - shaleW - porW;

            // Matrix (Gray)
            logCtx.fillStyle = '#374151';
            logCtx.fillRect(330, topPadding + y, matrixW, 2);
            // Shale (Brown/Orange)
            logCtx.fillStyle = '#854d0e';
            logCtx.fillRect(330 + matrixW, topPadding + y, shaleW, 2);
            // Porosity Fluids (Blue for Sw, Green for Hydrocarbon)
            let hydroW = porW * (1 - p.sw);
            let watW = porW * p.sw;
            logCtx.fillStyle = '#10b981'; // Hydrocarbon pay
            logCtx.fillRect(330 + matrixW + shaleW, topPadding + y, hydroW, 2);
            logCtx.fillStyle = '#2563eb'; // Water wet
            logCtx.fillRect(330 + matrixW + shaleW + hydroW, topPadding + y, watW, 2);
        }

        // Depth Probe Marker
        let activeY = topPadding + (currentDepth / maxD) * plotHeight;
        logCtx.strokeStyle = '#f43f5e';
        logCtx.lineWidth = 1.5;
        logCtx.beginPath();
        logCtx.moveTo(25, activeY);
        logCtx.lineTo(525, activeY);
        logCtx.stroke();
    }

    function drawCrossplot() {
        crossplotCtx.clearRect(0, 0, crossplotCanvas.width, crossplotCanvas.height);
        let w = crossplotCanvas.width;
        let h = crossplotCanvas.height;
        let padding = 25;

        // Draw axes
        crossplotCtx.strokeStyle = '#4b5563';
        crossplotCtx.lineWidth = 1;
        crossplotCtx.strokeRect(padding, padding, w - padding * 2, h - padding * 2);

        // Axes Labels
        crossplotCtx.fillStyle = '#9ca3af';
        crossplotCtx.font = '9px monospace';
        crossplotCtx.fillText('\u03A6 (Porosity) 0% -> 45%', padding, h - 8);

        crossplotCtx.save();
        crossplotCtx.translate(10, h/2 + 30);
        crossplotCtx.rotate(-Math.PI/2);
        crossplotCtx.fillText('Resistivity (Log)', 0, 0);
        crossplotCtx.restore();

        // Plot background clusters (Shale, Pay, Wet Sands)
        for (let d = 0; d < 3000; d += 25) {
            let p = getSubsurfaceParameters(d);
            let cx = padding + (p.phi / 0.45) * (w - padding * 2);
            let cy = h - padding - (Math.log10(p.res + 0.9) / 2.1) * (h - padding * 2);

            if (p.vsh > vshCutoff) {
                crossplotCtx.fillStyle = '#ef444455'; // Shale Cluster
            } else if (p.sw < 0.4) {
                crossplotCtx.fillStyle = '#10b981aa'; // Hydrocarbon cluster
            } else {
                crossplotCtx.fillStyle = '#3b82f6aa'; // Wet Sand cluster
            }
            crossplotCtx.beginPath();
            crossplotCtx.arc(cx, cy, 2.5, 0, 2 * Math.PI);
            crossplotCtx.fill();
        }

        // Draw active probe point on crossplot
        let currentP = getSubsurfaceParameters(currentDepth);
        let activeCx = padding + (currentP.phi / 0.45) * (w - padding * 2);
        let activeCy = h - padding - (Math.log10(currentP.res + 0.9) / 2.1) * (h - padding * 2);

        crossplotCtx.strokeStyle = '#ffffff';
        crossplotCtx.lineWidth = 2;
        crossplotCtx.beginPath();
        crossplotCtx.arc(activeCx, activeCy, 7, 0, 2 * Math.PI);
        crossplotCtx.stroke();

        crossplotCtx.fillStyle = '#f43f5e';
        crossplotCtx.beginPath();
        crossplotCtx.arc(activeCx, activeCy, 4, 0, 2 * Math.PI);
        crossplotCtx.fill();
    }

    function update() {
        let p = getSubsurfaceParameters(currentDepth);

        // Update labels
        document.getElementById('depthVal').textContent = currentDepth + ' m';
        document.getElementById('rwVal').textContent = rw.toFixed(2) + ' \u03A9m';
        document.getElementById('mVal').textContent = archieM.toFixed(1);
        document.getElementById('vshCutoffVal').textContent = (vshCutoff * 100).toFixed(0) + '%';

        document.getElementById('lithVal').textContent = p.lith;
        document.getElementById('vshVal').textContent = (p.vsh * 100).toFixed(1) + '%';
        document.getElementById('porVal').textContent = (p.phie * 100).toFixed(1) + '%';
        document.getElementById('swVal').textContent = (p.sw * 100).toFixed(1) + '%';

        // Classify current zone reservoir potential
        let badge = document.getElementById('zoneClassification');
        if (p.vsh > vshCutoff) {
            badge.textContent = 'Non-Reservoir (High Shale)';
            badge.style.backgroundColor = '#991b1b';
            badge.style.color = '#fee2e2';
        } else if (p.sw < 0.35) {
            badge.textContent = 'Hydrocarbon Pay Candidate';
            badge.style.backgroundColor = '#065f46';
            badge.style.color = '#d1fae5';
        } else {
            badge.textContent = 'Wet Reservoir Rock';
            badge.style.backgroundColor = '#1e3a8a';
            badge.style.color = '#dbeafe';
        }

        drawLogs();
        drawCrossplot();
    }

    // Event Listeners
    document.getElementById('depthRange').addEventListener('input', (e) => {
        currentDepth = +e.target.value;
        update();
    });
    document.getElementById('rwRange').addEventListener('input', (e) => {
        rw = +e.target.value;
        update();
    });
    document.getElementById('mRange').addEventListener('input', (e) => {
        archieM = +e.target.value;
        update();
    });
    document.getElementById('vshCutoffRange').addEventListener('input', (e) => {
        vshCutoff = (+e.target.value) / 100;
        update();
    });

    // Initialize
    update();
</script>

</body>
</html>"""))